# SAL_W_WARD DEDUPLICATION

**Tess Vu**

This notebook goes over Jill's ArcGIS Pro spatial join output `sal_w_ward` to address duplicates.

In [11]:
import geopandas as gpd
import pandas as pd
import os

In [12]:
# LOAD THE ORIGINAL SHAPEFILE.
sal_raw = gpd.read_file("data/sal_w_ward_new/sal_w_ward_new.shp")

print(f"ORIGINAL: {sal_raw.shape[0]} features")
print(f"Unique EA_CODEs: {sal_raw['EA_CODE'].nunique()}")
print(f"Duplicate EA_CODE rows: {sal_raw.shape[0] - sal_raw['EA_CODE'].nunique()}")
print()
print(f"AREA column range: {sal_raw['AREA'].min():.2f} to {sal_raw['AREA'].max():.2f}")
print(f"PERCENTAGE column range: {sal_raw['PERCENTAGE'].min():.4f} to {sal_raw['PERCENTAGE'].max():.4f}")

ORIGINAL: 39177 features
Unique EA_CODEs: 38380
Duplicate EA_CODE rows: 797

AREA column range: 1550.18 to 623519119.33
PERCENTAGE column range: 26.4165 to 100.0001


In [13]:
# PRE-DEDUPLICATION CROSS-CHECK 1: DUPLICATE STRUCTURE.
# Isolate all rows whose EA_CODE appears more than once.
# Each duplicated EA_CODE should appear exactly twice (one SAL straddling
# two wards), but checking for cases with 3+ occurrences is important.
dup_counts = sal_raw.groupby("EA_CODE").size()
n_appear_once   = (dup_counts == 1).sum()
n_appear_twice  = (dup_counts == 2).sum()
n_appear_more   = (dup_counts > 2).sum()

print("DUPLICATE OCCURRENCE COUNTS")
print(f"  EA_CODEs appearing exactly once  : {n_appear_once}")
print(f"  EA_CODEs appearing exactly twice : {n_appear_twice}")
print(f"  EA_CODEs appearing 3 or more times: {n_appear_more}")
if n_appear_more > 0:
    print(f"  WARNING: EA_CODEs with 3+ rows (inspect these manually):")
    print(dup_counts[dup_counts > 2].head(10))
print()

DUPLICATE OCCURRENCE COUNTS
  EA_CODEs appearing exactly once  : 37610
  EA_CODEs appearing exactly twice : 748
  EA_CODEs appearing 3 or more times: 22
EA_CODE
50410146.0    4
52410457.0    3
52910140.0    3
56810066.0    3
57510047.0    3
57510086.0    3
57510098.0    3
57510169.0    3
57510219.0    3
57510224.0    3
dtype: int64



In [15]:
# PRE-DEDUPLICATION CROSS-CHECK 2: WARD ASSIGNMENTS DIFFER BETWEEN DUPLICATES.
# The working hypothesis is that duplicates arise from SALs straddling
# two wards, meaning each duplicate pair should have two distinct WardID
# values. If WardIDs are identical within a duplicate group, the duplicate
# has a different cause and needs separate handling.
dup_ea_codes = dup_counts[dup_counts > 1].index
dup_rows = sal_raw[sal_raw["EA_CODE"].isin(dup_ea_codes)].copy()

ward_col = "census_war"

if ward_col in dup_rows.columns:
    ward_per_dup = dup_rows.groupby("EA_CODE")[ward_col].nunique()
    same_ward = (ward_per_dup == 1).sum()
    diff_ward = (ward_per_dup > 1).sum()

    print("WARD ASSIGNMENT CHECK ACROSS DUPLICATE PAIRS")
    print(f"  Duplicate groups with different wards (expected): {diff_ward}")
    print(f"  Duplicate groups with same ward (unexpected):     {same_ward}")

    if same_ward > 0:
        same_ward_codes = ward_per_dup[ward_per_dup == 1].index[:5]
        print(f"  Sample same-ward duplicates (inspect these):")
        print(dup_rows[dup_rows["EA_CODE"].isin(same_ward_codes)][["EA_CODE", ward_col, "AREA", "PERCENTAGE"]])
else:
    print(f"WARNING: Ward column '{ward_col}' not found. Available columns:")
    print([c for c in sal_raw.columns if "ward" in c.lower() or "WAR" in c])
print()

WARD ASSIGNMENT CHECK ACROSS DUPLICATE PAIRS
  Duplicate groups with different wards (expected): 0
  Duplicate groups with same ward (unexpected):     770
  Sample same-ward duplicates (inspect these):
        EA_CODE census_war          AREA  PERCENTAGE
68   50310126.0   52103004  8.946239e+06   99.999988
69   50310126.0   52103004  8.946239e+06   99.999988
85   50310193.0   52103003  2.679076e+06  100.000000
86   50310193.0   52103003  2.679076e+06  100.000000
172  50310099.0   52103019  1.020005e+06   93.826704
173  50310099.0   52103019  1.020005e+06   93.826704
197  50310063.0   52103012  1.026250e+07   90.546234
198  50310063.0   52103012  1.026250e+07   90.546234
256  50310086.0   52103018  1.189960e+06   99.387696
257  50310086.0   52103018  1.189960e+06   99.387696



In [14]:
# DIAGNOSTIC: FULL INSPECTION OF EA_CODEs WITH 3 OR MORE OCCURRENCES.
# Check 1 flagged 22 such EA_CODEs. This cell prints all of them and
# shows every row, to confirm whether the extra copies are also
# identical duplicates or carry distinct attribute values.
high_dup_codes = dup_counts[dup_counts > 2].index
print(f"ALL EA_CODEs WITH 3+ OCCURRENCES: {len(high_dup_codes)} total")
print()

high_dup_rows = sal_raw[sal_raw["EA_CODE"].isin(high_dup_codes)].copy()

inspect_cols = ["EA_CODE", ward_col, "AREA", "PERCENTAGE"]

for ea_code, group in high_dup_rows.groupby("EA_CODE"):
    n = len(group)
    n_unique_wards = group[ward_col].nunique()
    n_unique_areas = group["AREA"].nunique()
    n_unique_pcts  = group["PERCENTAGE"].nunique()
    n_unique_geoms = group.geometry.area.nunique()

    print(f"EA_CODE: {ea_code}  |  occurrences: {n}  |  unique wards: {n_unique_wards}"
          f"  |  unique AREA values: {n_unique_areas}  |  unique PERCENTAGE values: {n_unique_pcts}"
          f"  |  unique geometry areas: {n_unique_geoms}")
    print(group[inspect_cols].to_string(index = True))
    print()

ALL EA_CODEs WITH 3+ OCCURRENCES: 22 total

EA_CODE: 50410146.0  |  occurrences: 4  |  unique wards: 1  |  unique AREA values: 1  |  unique PERCENTAGE values: 1  |  unique geometry areas: 1
        EA_CODE census_war          AREA  PERCENTAGE
286  50410146.0   52104002  1.240001e+08   99.935694
287  50410146.0   52104002  1.240001e+08   99.935694
288  50410146.0   52104002  1.240001e+08   99.935694
289  50410146.0   52104002  1.240001e+08   99.935694

EA_CODE: 52410457.0  |  occurrences: 3  |  unique wards: 1  |  unique AREA values: 1  |  unique PERCENTAGE values: 1  |  unique geometry areas: 1
         EA_CODE census_war          AREA  PERCENTAGE
1457  52410457.0   52502006  1.450021e+06       100.0
1458  52410457.0   52502006  1.450021e+06       100.0
1459  52410457.0   52502006  1.450021e+06       100.0

EA_CODE: 52910140.0  |  occurrences: 3  |  unique wards: 1  |  unique AREA values: 1  |  unique PERCENTAGE values: 1  |  unique geometry areas: 1
         EA_CODE census_war        

In [5]:
# PRE-DEDUPLICATION CROSS-CHECK 3: AREA TIES.
# The deduplication strategy keeps the row with the largest AREA.
# If two rows for the same EA_CODE have identical AREA values, the
# tie-break is arbitrary (first in sort order). Flag any such ties.
area_per_dup = dup_rows.groupby("EA_CODE")["AREA"]
tied = dup_rows[
    dup_rows.duplicated(subset = ["EA_CODE", "AREA"], keep = False)
]["EA_CODE"].nunique()

print("AREA TIE CHECK")
print(f"  Duplicate EA_CODEs with tied AREA values: {tied}")
if tied > 0:
    tied_codes = dup_rows[
        dup_rows.duplicated(subset = ["EA_CODE", "AREA"], keep = False)
    ]["EA_CODE"].unique()[:5]
    print(f"  Sample tied EA_CODEs (tie-break will be arbitrary):")
    print(dup_rows[dup_rows["EA_CODE"].isin(tied_codes)][["EA_CODE", ward_col, "AREA", "PERCENTAGE"]])
print()

AREA TIE CHECK
  Duplicate EA_CODEs with tied AREA values: 770
  Sample tied EA_CODEs (tie-break will be arbitrary):
        EA_CODE census_war          AREA  PERCENTAGE
36   50310219.0   52103004  1.762411e+07   57.385730
37   50310219.0   52103004  1.762411e+07   57.385730
68   50310126.0   52103004  8.946239e+06   99.999988
69   50310126.0   52103004  8.946239e+06   99.999988
85   50310193.0   52103003  2.679076e+06  100.000000
86   50310193.0   52103003  2.679076e+06  100.000000
172  50310099.0   52103019  1.020005e+06   93.826704
173  50310099.0   52103019  1.020005e+06   93.826704
197  50310063.0   52103012  1.026250e+07   90.546234
198  50310063.0   52103012  1.026250e+07   90.546234



In [6]:
# PRE-DEDUPLICATION CROSS-CHECK 4: GEOMETRY CONSISTENCY.
# Each row for the same EA_CODE should carry the same polygon geometry,
# since the SAL boundary does not change across ward assignments.
# Compare geometry area (shapely .area) across duplicate pairs.
# Large discrepancies indicate geometry corruption or a data join error.
dup_rows["geom_area"] = dup_rows.geometry.area
geom_spread = dup_rows.groupby("EA_CODE")["geom_area"].agg(["min", "max"])
geom_spread["rel_diff"] = (geom_spread["max"] - geom_spread["min"]) / geom_spread["max"]

# Flag EA_CODEs where geometry area differs by more than 1% across duplicates.
geom_mismatch = geom_spread[geom_spread["rel_diff"] > 0.01]

print("GEOMETRY CONSISTENCY CHECK (across duplicate pairs)")
print(f"  Duplicate groups with geometry area difference > 1%: {len(geom_mismatch)}")
if len(geom_mismatch) > 0:
    print(f"  WARNING: These EA_CODEs carry different geometries across duplicate rows:")
    print(geom_mismatch.head(10))
else:
    print(f"  All duplicate pairs carry consistent geometry.")
print()

GEOMETRY CONSISTENCY CHECK (across duplicate pairs)
  Duplicate groups with geometry area difference > 1%: 0
  All duplicate pairs carry consistent geometry.



In [7]:
# PRE-DEDUPLICATION CROSS-CHECK 5: WARD MATCH AGAINST pop_pred_final.
# After deduplication, the retained WardID for each EA_CODE should match
# the WardID in pop_pred_final.csv, which was built from the same spatial
# join using the same dominant-ward logic. A mismatch here means the
# deduplication strategy diverges from how the population model assigned wards.
pop = pd.read_csv("data/pop_pred_final.csv")

# Build a lookup from pop_pred_final: EA_CODE -> WardID.
pop_ward_lookup = pop.set_index("EA_CODE")["WardID"]

# Simulate deduplication to get the retained WardID per EA_CODE.
sal_sorted = sal_raw.sort_values("AREA", ascending = False)
retained = sal_sorted.drop_duplicates(subset = ["EA_CODE"], keep = "first")[["EA_CODE", ward_col]].copy()
retained = retained.rename(columns = {ward_col: "shp_WardID"})
retained["shp_WardID"] = retained["shp_WardID"].astype(str)

# Align types for comparison.
pop_ward_series = pop_ward_lookup.astype(str)
retained["pop_WardID"] = retained["EA_CODE"].map(pop_ward_series)

# Only check EA_CODEs that were actually duplicated (single-occurrence
# EA_CODEs are trivially matched).
retained_dups = retained[retained["EA_CODE"].isin(dup_ea_codes)].copy()
ward_matches   = (retained_dups["shp_WardID"] == retained_dups["pop_WardID"]).sum()
ward_mismatches = (retained_dups["shp_WardID"] != retained_dups["pop_WardID"]).sum()

print("WARD MATCH AGAINST pop_pred_final (duplicated EA_CODEs only)")
print(f"  Retained WardID matches pop_pred_final: {ward_matches} / {len(retained_dups)}")
print(f"  Mismatches: {ward_mismatches}")

if ward_mismatches > 0:
    mismatch_rows = retained_dups[retained_dups["shp_WardID"] != retained_dups["pop_WardID"]].head(10)
    print()
    print("  Sample mismatches (shp_WardID vs pop_WardID):")
    print(mismatch_rows.to_string(index = False))
    print()
    print("  RECOMMENDATION: If mismatches are material, consider using PERCENTAGE")
    print("  instead of AREA as the deduplication key, or adopt the WardID from")
    print("  pop_pred_final directly via a post-deduplication merge.")
print()

WARD MATCH AGAINST pop_pred_final (duplicated EA_CODEs only)
  Retained WardID matches pop_pred_final: 770 / 770
  Mismatches: 0



In [ ]:
# DEDUPLICATE: sort descending by AREA, keep first (dominant ward) per EA_CODE.
sal_dedup = (
    sal_raw
    .sort_values("AREA", ascending = False)
    .drop_duplicates(subset = ["EA_CODE"], keep = "first")
    .sort_index()
    .copy()
)

print(f"AFTER DEDUPLICATION: {sal_dedup.shape[0]} features")
print(f"Unique EA_CODEs: {sal_dedup['EA_CODE'].nunique()}")
print(f"Duplicates remaining: {sal_dedup.shape[0] - sal_dedup['EA_CODE'].nunique()}")

AFTER DEDUPLICATION: 38380 features
Unique EA_CODEs: 38380
Duplicates remaining: 0


In [9]:
# POST-DEDUPLICATION CROSS-CHECK: EA_CODE MEMBERSHIP.
pop_codes = set(pop["EA_CODE"].unique())
sal_codes = set(sal_dedup["EA_CODE"].unique())

only_in_sal = sal_codes - pop_codes
only_in_pop = pop_codes - sal_codes

print("POST-DEDUPLICATION EA_CODE MEMBERSHIP CHECK")
print(f"  EA_CODEs in shapefile but NOT in pop_pred_final: {len(only_in_sal)}")
print(f"  EA_CODEs in pop_pred_final but NOT in shapefile: {len(only_in_pop)}")

if len(only_in_sal) > 0:
    print(f"  Sample (shapefile-only): {list(only_in_sal)[:5]}")
if len(only_in_pop) > 0:
    print(f"  Sample (pop-only): {list(only_in_pop)[:5]}")

# Confirm final geometry is valid (no null or empty geometries).
null_geom = sal_dedup.geometry.isna().sum()
empty_geom = sal_dedup.geometry.is_empty.sum()
print()
print(f"  Null geometries : {null_geom}")
print(f"  Empty geometries: {empty_geom}")

# Confirm CRS was preserved.
print(f"  CRS: {sal_dedup.crs}")

POST-DEDUPLICATION EA_CODE MEMBERSHIP CHECK
  EA_CODEs in shapefile but NOT in pop_pred_final: 0
  EA_CODEs in pop_pred_final but NOT in shapefile: 0

  Null geometries : 0
  Empty geometries: 0
  CRS: EPSG:32735


In [10]:
# SAVE DEDUPLICATED SHAPEFILE.
output_dir = "data/sal_w_ward_dedup"
os.makedirs(output_dir, exist_ok = True)

output_path = os.path.join(output_dir, "sal_w_ward_dedup.shp")
sal_dedup.to_file(output_path)

print(f"SAVED: {output_path}")
print(f"Final shape: {sal_dedup.shape}")

SAVED: data/sal_w_ward_dedup\sal_w_ward_dedup.shp
Final shape: (38380, 70)


## NOTES

Duplicate rows carry identical ward, area, and geometry values, indicating a data artifact from the spatial join construction rather than a SAL-to-ward boundary overlap